# FraudGraph: Temporal Drift & Adaptation

**Exploratory investigation, not an independent fresh test.** The original project already identified a collapse near time step 43. Generate tables using `python scripts/run_temporal_research.py` (and optionally `--include-posthoc-test`) first.

**Questions:** Did input features, known-label prevalence, prediction scores or ranking performance change? Are graph components acting as cohort signatures? Can a time-aware learner hold up better?

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT = ROOT / 'reports' / 'exploratory'
s = pd.read_csv(OUT / 'label_score_drift_by_time.csv')
s.head()

## Label prevalence and ranking performance
Average Precision (AP) is prevalence-sensitive; compare AP with the known-illicit base rate and show the absolute number of labeled positives in each bucket. Unknown-label transactions are excluded from labeled AP.

In [ ]:
ax = s.plot(x='time_step', y=['known_illicit_prevalence', 'average_precision_labeled'], marker='o', figsize=(10, 4), title='Known-label prevalence vs AP, original model')
ax.axvline(43, linestyle='--', alpha=.5)
ax.set_ylabel('Fraction / AP'); ax.grid(alpha=.2)
plt.show()
display(s[['time_step', 'labeled', 'known_illicit', 'known_label_coverage', 'ap_above_prevalence']])

## Input distribution shifts
PSI and Wasserstein measure different aspects of input shift; neither proves concept drift. Compare the original transaction size and graph features.

In [ ]:
f = pd.read_csv(OUT / 'feature_drift_by_time.csv')
for feature in ['size','fees','g_component_size','g_pagerank_scaled']:
    subset = f[f.feature == feature]
    plt.plot(subset.time_step, subset.psi, marker='o', label=feature)
plt.legend(); plt.xlabel('Time step'); plt.ylabel('PSI vs train history'); plt.title('Feature drift'); plt.grid(alpha=.2); plt.show()
display(f.loc[f.time_step == 43].sort_values('psi', ascending=False).head(12))

## Temporal model selection and graph ablations
Selection uses validation folds only. Changes on the already-inspected test steps are post-hoc research. Do not optimize the model a second time using those test results.

In [ ]:
leaderboard = pd.read_csv(OUT / 'adaptive_validation_leaderboard.csv')
ablations = pd.read_csv(OUT / 'graph_ablation_by_time.csv')
display(leaderboard.head(12))
display(ablations.groupby(['feature_group', 'period'])['ap'].agg(['mean','min','count']))
p = OUT / 'adaptive_posthoc_test_by_time.csv'
if p.exists():
    posthoc = pd.read_csv(p)
    display(posthoc[['time_step','strategy','last_training_step','known_illicit','prevalence','ap']])
else:
    print('Post-hoc test replay not run. Opt in explicitly via --include-posthoc-test.')

## Interpretation notes
1. Did known-label prevalence or coverage change at the same time as AP?
2. Does transaction `size` drift, or does its relationship with the target seem to change?
3. Does removing component signatures help some future-like buckets despite losing pooled AP?
4. Are results stable to assumed label delays of 2, 4 or more buckets?
5. Without genuine label-maturation timestamps and fresh future data, what conclusions remain unsupported?